# CHỦ ĐỀ 4: SỨC KHỎE PHỤ NỮ VÀ BÌNH ĐẲNG GIỚI

## 1. Phân tích cơ bản về dữ liệu

### 1.1. Giới thiệu tổng quan về dataset

Dataset được sử dụng trong bài phân tích này được thu thập từ **World Development Indicators (WDI)** do Ngân hàng Thế giới (World Bank) cung cấp. Dữ liệu bao gồm thông tin của **8 quốc gia** đa dạng về mức thu nhập và khu vực địa lý, trong giai đoạn từ **2010 đến 2023**.

Các quốc gia được phân tích: **Germany, India, Indonesia, Japan, Korea (Rep.), Thailand, United States, Viet Nam.**

**Lý do chọn 8 quốc gia này:** nhằm tạo bối cảnh so sánh đa chiều giữa các nền kinh tế phát triển (Mỹ, Đức, Nhật, Hàn) và đang phát triển (Ấn Độ, Indonesia, Thái Lan, Việt Nam) để đánh giá hiệu quả chi tiêu y tế đối với sức khỏe phụ nữ và phân tích bình đẳng giới trong cấu trúc dân số.

### 1.2. Cấu trúc dữ liệu và các trường dữ liệu

| Cột | Mô tả |
|-----|--------|
| Country Name | Tên quốc gia |
| Country Code | Mã quốc gia (ISO 3-letter) |
| Series Name | Tên chỉ số |
| Series Code | Mã chỉ số |
| 2010 [YR2010] ... 2023 [YR2023] | Giá trị chỉ số theo từng năm |

Mỗi quốc gia có **3 chỉ số (Series):**

1. **`SH.XPD.CHEX.PC.CD`** — Current health expenditure per capita (current US$): Chi tiêu y tế bình quân đầu người theo giá hiện hành (USD).
2. **`SP.DYN.LE00.FE.IN`** — Life expectancy at birth, female (years): Tuổi thọ trung bình khi sinh của nữ giới (năm).
3. **`SP.POP.TOTL.FE.ZS`** — Population, female (% of total population): Tỷ lệ dân số nữ so với tổng dân số (%).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Cài đặt tham số đồ họa
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

# 1. Đọc dữ liệu từ file lớn và lọc năm từ đầu
DATA_PATH = '../data3/dataset.csv' # Đường dẫn tới dataset tổng hợp
df_raw_full = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

# Danh sách các chỉ số cần dùng cho Chủ đề 4 / Sức khỏe phụ nữ
HEALTH_INDICATORS = [
    'SH.XPD.CHEX.PC.CD', 
    'SP.DYN.LE00.FE.IN', 
    'SP.POP.TOTL.FE.ZS'
]

# Lựa chọn 8 quốc gia
SELECTED_COUNTRIES = ['DEU', 'IND', 'IDN', 'JPN', 'KOR', 'THA', 'USA', 'VNM']

# 2. Xử lý dữ liệu: Lọc các cột cần thiết (bao gồm luôn lọc theo năm 2010-2023 từ Wide format)
year_cols = [f"{year} [YR{year}]" for year in range(2010, 2024)]
base_cols = ['Country Name', 'Country Code', 'Series Name', 'Series Code']

# Lọc dataset (chỉ số, quốc gia và năm)
df_raw = df_raw_full[
    (df_raw_full['Series Code'].isin(HEALTH_INDICATORS)) & 
    (df_raw_full['Country Code'].isin(SELECTED_COUNTRIES))
][base_cols + year_cols].copy()

# Chuyển từ Wide sang Long format
df_long = df_raw.melt(
    id_vars=base_cols,
    value_vars=year_cols,
    var_name='YearRaw', value_name='Value'
)

# Lấy ra con số năm (VD: "2010 [YR2010]" -> 2010)
df_long['Year'] = df_long['YearRaw'].str.extract(r'(\d{4})').astype(int)
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')
df_long = df_long.sort_values(by=['Country Code', 'Series Code', 'Year']).reset_index(drop=True)

print(f"Kích thước bảng ban đầu sau khi lọc: {df_raw.shape[0]} dòng × {df_raw.shape[1]} cột")
print(f"Số bản ghi sau khi chuyển đổi: {len(df_long)}")
print(f"Số giá trị thiếu (NaN) trước nội suy: {df_long['Value'].isna().sum()}")
print(f"Các quốc gia: {sorted(df_long['Country Name'].unique())}")
df_long.head()

In [ ]:
# 3. Kiểm tra dữ liệu thiếu chi tiết
missing_detail = df_long[df_long['Value'].isna()][['Country Name', 'Series Code', 'Year']]
if len(missing_detail) > 0:
    print("Các ô dữ liệu bị thiếu (NaN):")
    print(missing_detail.to_string(index=False))
else:
    print("Không có dữ liệu bị thiếu.")

In [ ]:
# 4. Xử lý dữ liệu thiếu bằng nội suy tuyến tính trong nhóm (quốc gia, chỉ số)
df_clean = df_long.copy()
df_clean['Value'] = df_clean.groupby(['Country Code', 'Series Code'])['Value'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

print(f"Số giá trị thiếu sau xử lý: {df_clean['Value'].isna().sum()}")

In [ ]:
# 5. Thống kê cơ bản theo từng chỉ số
for code_val in sorted(df_clean['Series Code'].unique()):
    subset = df_clean[df_clean['Series Code'] == code_val]
    name = subset['Series Name'].iloc[0]
    print(f"\n--- {code_val}: {name} ---")
    print(subset['Value'].describe().to_string())

## Xử lý dữ liệu thiếu

Trong dataset, nhóm tiến hành kiểm tra dữ liệu cho 8 quốc gia trong giai đoạn 2010–2023 với 3 chỉ số được lựa chọn. Kết quả cho thấy dữ liệu đầy đủ, liên tục và không có giá trị bị thiếu trong phạm vi phân tích.

## Phương pháp xử lý

Do không tồn tại giá trị thiếu, nhóm không áp dụng bất kỳ phương pháp xử lý hay ước lượng nào (như nội suy tuyến tính). Toàn bộ dữ liệu được sử dụng trực tiếp cho quá trình phân tích và trực quan hóa.

## Lý do lựa chọn
Tính toàn vẹn dữ liệu cao → không cần can thiệp hay biến đổi dữ liệu gốc
Đảm bảo độ chính xác → tránh sai lệch do các phương pháp ước lượng
Phù hợp với time-series → giữ nguyên xu hướng thực tế của dữ liệu theo thời gian

---

## 2. Xác định mục tiêu và lựa chọn trường dữ liệu

---

### 2.1. Mục tiêu 4: Đánh giá hiệu quả của chi tiêu y tế đối với sức khỏe phụ nữ

**Tiêu chí SMART:**
- **S (Specific):** Đánh giá mối quan hệ giữa chi tiêu y tế và tuổi thọ nữ.
- **M (Measurable):** Dựa trên số liệu liên tục qua các năm của biến `SH.XPD.CHEX.PC.CD` (health expenditure per capita) và `SP.DYN.LE00.FE.IN` (life expectancy female).
- **A (Achievable):** Data đủ tốt cho 8 quốc gia giai đoạn 2010–2023.
- **R (Relevant):** Kiểm tra xem đầu tư y tế có thực sự mang lại hiệu quả cải thiện sức khỏe phụ nữ hay không.
- **T (Time-bound):** Từ năm 2010 đến 2023.

### Trường dữ liệu sử dụng

1. **`SH.XPD.CHEX.PC.CD`** — Current health expenditure per capita (current US$)
   - Chỉ số đo lường tổng chi tiêu y tế bình quân đầu người, bao gồm cả chi tiêu công và tư nhân.
   - Phản ánh mức độ đầu tư thực tế của quốc gia vào hệ thống chăm sóc sức khỏe.

2. **`SP.DYN.LE00.FE.IN`** — Life expectancy at birth, female (years)
   - Tuổi thọ kỳ vọng khi sinh của nữ giới, thể hiện số năm trung bình một bé gái được dự đoán sẽ sống dưới điều kiện hiện tại.
   - Là một chỉ số tổng hợp phản ánh tình trạng sức khỏe của phụ nữ, chịu ảnh hưởng bởi nhiều yếu tố. Trong phạm vi của mục tiêu này, chỉ số này được sử dụng để xem xét mối liên hệ với chi tiêu y tế.

### Lý do lựa chọn metrics

Cặp biến này được chọn vì cho phép kiểm tra giả thuyết: **"Chi tiêu y tế cao hơn → tuổi thọ nữ cao hơn"**. Nếu mối quan hệ này yếu hoặc không tuyến tính, thì điều này gợi ý rằng hiệu quả sử dụng chi tiêu y tế có thể quan trọng hơn quy mô chi tiêu đơn thuần.

### Biểu đồ 1: Scatter Plot + Regression Line — Mối quan hệ chi tiêu y tế và tuổi thọ nữ

**Lý do chọn biểu đồ Scatter Plot + Regression Line:**
Đây là phương pháp trực quan hóa **chuẩn mực trong academic** để đánh giá mối tương quan giữa hai biến liên tục. Mỗi điểm đại diện cho 1 quốc gia tại 1 năm, giúp ta thấy rõ:
- **Correlation:** mối tương quan tổng thể giữa chi tiêu và tuổi thọ
- **Outliers:** quốc gia nào chi nhiều nhưng hiệu quả thấp
- **Clustering:** nhóm quốc gia có đặc điểm tương tự

In [ ]:
# ====== BIỂU ĐỒ 1: SCATTER PLOT + REGRESSION — Chi tiêu y tế vs Tuổi thọ nữ ======

# Chuẩn bị dữ liệu: pivot ra các biến
pivot_df = df_clean.pivot_table(
    index=['Country Name', 'Country Code', 'Year'],
    columns='Series Code',
    values='Value'
).reset_index()
pivot_df.columns.name = None

# Đổi tên cột cho dễ đọc
pivot_df = pivot_df.rename(columns={
    'SH.XPD.CHEX.PC.CD': 'health_exp',
    'SP.DYN.LE00.FE.IN': 'life_exp_female',
    'SP.POP.TOTL.FE.ZS': 'female_pop'
})

scatter_df = pivot_df.dropna(subset=['health_exp', 'life_exp_female']).copy()

# Tính hệ số tương quan Pearson
r_value, p_value = stats.pearsonr(scatter_df['health_exp'], scatter_df['life_exp_female'])

fig, ax = plt.subplots(figsize=(14, 8))

# Scatter plot với màu theo quốc gia
palette = sns.color_palette('tab10', n_colors=len(scatter_df['Country Name'].unique()))
sns.scatterplot(data=scatter_df, x='health_exp', y='life_exp_female',
                hue='Country Name', palette=palette, s=100, alpha=0.8, edgecolor='white', ax=ax)

# Regression line tổng thể
sns.regplot(data=scatter_df, x='health_exp', y='life_exp_female',
            scatter=False, color='red', line_kws={'linewidth': 2.5, 'linestyle': '--'}, ax=ax)

# Ghi chú thống kê
ax.text(0.02, 0.96,
        f'Pearson r = {r_value:.4f} (p = {p_value:.6f})\n'
        f'Số quan sát: {len(scatter_df)}',
        transform=ax.transAxes, fontsize=12, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.title('Mối quan hệ giữa Chi tiêu Y tế và Tuổi thọ Nữ (2010–2023)\n(Mỗi điểm = 1 quốc gia tại 1 năm)',
          fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Chi tiêu y tế bình quân đầu người (USD)', fontsize=13, fontweight='bold')
plt.ylabel('Tuổi thọ nữ (năm)', fontsize=13, fontweight='bold')
ax.legend(title='Quốc gia', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"KẾT QUẢ PHÂN TÍCH TƯƠNG QUAN")
print(f"{'='*60}")
print(f"Hệ số tương quan Pearson r = {r_value:.4f}")
print(f"p-value = {p_value:.6f}")
print(f"Số quan sát: {len(scatter_df)}")
if abs(r_value) < 0.3:
    print("→ Tương quan YẾU (|r| < 0.3)")
elif abs(r_value) < 0.5:
    print("→ Tương quan TRUNG BÌNH (0.3 ≤ |r| < 0.5)")
elif abs(r_value) < 0.7:
    print("→ Tương quan KHÁ (0.5 ≤ |r| < 0.7)")
else:
    print("→ Tương quan MẠNH (|r| ≥ 0.7)")

## Phân tích biểu đồ Scatter Plot + Regression

### 1. Xu hướng tổng thể

Nhìn chung, biểu đồ cho thấy một **mối tương quan dương ở mức trung bình** giữa chi tiêu y tế và tuổi thọ nữ (Pearson r ≈ 0.47), tức là các quốc gia chi tiêu nhiều hơn **có xu hướng** có tuổi thọ nữ cao hơn.  

Tuy nhiên, mối quan hệ này **không tuyến tính hoàn hảo**, thể hiện qua sự phân tán đáng kể của các điểm dữ liệu quanh đường hồi quy.

---

### 2. Dấu hiệu "hiệu suất giảm dần"

Ở nhóm quốc gia có mức chi tiêu thấp (India, Indonesia, Viet Nam, Thailand), tuổi thọ nữ có xu hướng tăng khi chi tiêu tăng, cho thấy chi tiêu y tế đóng vai trò quan trọng trong việc cải thiện sức khỏe ở giai đoạn đầu.  

Trong khi đó, ở nhóm quốc gia có mức chi tiêu cao (Germany, Japan, Korea, United States), tuổi thọ nữ duy trì ở mức cao (~81 - 88 tuổi) nhưng không gia tăng đáng kể theo mức chi tiêu. 

→ Điều này **gợi ý khả năng tồn tại hiện tượng “hiệu suất giảm dần”**, tức là việc tăng chi tiêu ở mức cao có thể không mang lại cải thiện tương ứng về kết quả sức khỏe.

---

### 3. Trường hợp đáng chú ý — United States

United States nổi bật với mức chi tiêu y tế cao nhất trong nhóm (khoảng $8,000–$13,000/người), trong khi tuổi thọ nữ chỉ dao động khoảng 79–81 năm, thấp hơn một số quốc gia có mức chi tiêu thấp hơn như Nhật Bản và Hàn Quốc.  

→ Điều này cho thấy rằng **mức chi tiêu cao không nhất thiết đi kèm với kết quả sức khỏe vượt trội**, và có thể tồn tại sự khác biệt về hiệu quả sử dụng nguồn lực giữa các quốc gia.

---

### 4. So sánh giữa các quốc gia phát triển

| Quốc gia | Chi tiêu y tế/người | Tuổi thọ nữ |
|----------|---------------------|-------------|
| Mỹ | ~$8,000 – $13,000 | ~79 – 81 |
| Nhật | ~$3,600 – $5,200 | ~86 – 88 |
| Hàn | ~$1,400 – $3,100 | ~84 – 87 |

→ Nhật Bản và Hàn Quốc đạt tuổi thọ nữ cao hơn dù mức chi tiêu thấp hơn đáng kể so với Hoa Kỳ.  

→ Điều này **củng cố nhận định rằng hiệu quả sử dụng chi tiêu y tế có thể đóng vai trò quan trọng**, thay vì chỉ xét quy mô chi tiêu.

---

### 5. Kết luận

Kết quả cho thấy rằng mặc dù tồn tại mối tương quan dương giữa chi tiêu y tế và tuổi thọ nữ, nhưng **mối quan hệ này không mạnh và không hoàn toàn tuyến tính**.  

→ Điều này cho thấy rằng **chi tiêu y tế cao không tự động dẫn đến kết quả sức khỏe tốt hơn**, và hiệu quả sử dụng nguồn lực có thể là yếu tố quan trọng cần xem xét.

---

### 6. Hàm ý chính sách cho Việt Nam

Đối với Việt Nam, kết quả này gợi ý rằng:

- Việc **tăng chi tiêu y tế là cần thiết**, nhưng  
- Quan trọng hơn là:
  - nâng cao **hiệu quả phân bổ nguồn lực**  
  - đầu tư vào **y tế dự phòng và chăm sóc ban đầu**  
  - đảm bảo mọi nhóm dân cư, đặc biệt là phụ nữ ở vùng nông thôn hoặc có thu nhập thấp, đều có khả năng tiếp cận dịch vụ y tế một cách đầy đủ và hợp lý

→ Điều này có thể giúp cải thiện sức khỏe phụ nữ **mà không cần tăng mạnh chi tiêu như các nước phát triển**.

### Biểu đồ 2: Line Chart — Chi tiêu y tế và tuổi thọ nữ theo từng quốc gia

**Lý do chọn biểu đồ Line Chart (Dual-axis):**
Line Chart giúp theo dõi **xu hướng theo thời gian** của từng quốc gia riêng biệt, với 2 trục:
- Trục trái: Chi tiêu y tế (USD/người)
- Trục phải: Tuổi thọ nữ (năm)

Điều này cho phép quan sát liệu khi chi tiêu tăng, tuổi thọ có tăng tương ứng hay không tại **từng quốc gia cụ thể**.

In [ ]:
# ====== BIỂU ĐỒ 2: LINE CHART — Chi tiêu y tế & Tuổi thọ nữ theo từng quốc gia ======

countries = sorted(scatter_df['Country Name'].unique())
n_countries = len(countries)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, country in enumerate(countries):
    ax1 = axes[idx]
    country_data = scatter_df[scatter_df['Country Name'] == country].sort_values('Year')
    
    # Trục trái: Health expenditure
    color1 = '#3498db'
    ax1.plot(country_data['Year'], country_data['health_exp'], 
             color=color1, marker='o', linewidth=2, markersize=4, label='Chi tiêu y tế')
    ax1.set_ylabel('Chi tiêu y tế (USD)', color=color1, fontsize=9)
    ax1.tick_params(axis='y', labelcolor=color1, labelsize=8)
    
    # Trục phải: Life expectancy
    ax2 = ax1.twinx()
    color2 = '#e74c3c'
    ax2.plot(country_data['Year'], country_data['life_exp_female'],
             color=color2, marker='s', linewidth=2, markersize=4, label='Tuổi thọ nữ')
    ax2.set_ylabel('Tuổi thọ nữ (năm)', color=color2, fontsize=9)
    ax2.tick_params(axis='y', labelcolor=color2, labelsize=8)
    
    ax1.set_title(country, fontsize=11, fontweight='bold')
    ax1.set_xlabel('Năm', fontsize=9)
    ax1.tick_params(axis='x', rotation=45, labelsize=7)
    
    # Legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=7)

plt.suptitle('Chi tiêu y tế và Tuổi thọ nữ theo từng quốc gia (2010–2023)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Phân tích biểu đồ Line Chart theo từng quốc gia:**

### 1. Nhóm quốc gia phát triển

- **Xu hướng tương đồng:** Germany ($4,500→$6,400; ~83), Japan ($3,600–$5,200; ~86–88), và Korea ($1,400→$3,000+; ~84→86+) đều có mức chi tiêu tương đối cao và tuổi thọ nữ duy trì ở mức cao, cho thấy khi đạt đến một ngưỡng phát triển nhất định, tuổi thọ nữ không thay đổi đáng kể dù chi tiêu tiếp tục tăng.

- **Sự khác biệt:** United States ($7,800→$13,400+; ~81→79→80) có mức chi tiêu cao nhất nhưng tuổi thọ nữ không vượt trội, thậm chí giảm trong một số năm, cho thấy mức chi tiêu cao không nhất thiết đi kèm với kết quả sức khỏe tốt hơn.

---

### 2. Nhóm quốc gia đang phát triển

- **Xu hướng tương đồng:** India ($45→$85; ~69→74), Indonesia ($86→$160; ~70→73), Thailand ($170→$370; ~79→81), và Viet Nam ($85→$197; ~78→80) đều có mức chi tiêu thấp đến trung bình và tuổi thọ nữ tăng rõ theo thời gian, cho thấy chi tiêu y tế có xu hướng gắn với cải thiện sức khỏe trong giai đoạn phát triển.

- **Sự khác biệt:** Thailand và Viet Nam có mức chi tiêu cao hơn trong nhóm và đồng thời đạt mức tuổi thọ nữ cao hơn so với India và Indonesia, cho thấy sự khác biệt về mức độ phát triển và kết quả sức khỏe trong cùng nhóm quốc gia đang phát triển.

---

### Kết luận

Sự khác biệt giữa hai nhóm quốc gia cho thấy chi tiêu y tế có tác động không đồng đều: trong khi đóng vai trò rõ rệt ở các quốc gia đang phát triển, ở các quốc gia phát triển, việc gia tăng chi tiêu không đảm bảo cải thiện tương ứng về tuổi thọ, gợi ý vai trò của hiệu quả sử dụng nguồn lực.

### 3. Tác động COVID-19 (2020–2021):

Hầu hết các quốc gia ghi nhận **sụt giảm tuổi thọ nữ** trong năm 2020–2021, đặc biệt:
- **India:** giảm mạnh từ 72.3 → 68.8 năm (2021)
- **Indonesia:** giảm từ 72.4 → 69.4 năm (2021)
- **Mỹ:** giảm từ 81.4 → 79.3 năm (2021)

→ Đại dịch cho thấy tác động đến tuổi thọ nữ khác nhau giữa các quốc gia, gợi ý rằng chi tiêu y tế không phải là yếu tố duy nhất, mà hiệu quả tổ chức và khả năng ứng phó của hệ thống y tế cũng đóng vai trò quan trọng.

---

## NHÓM 4: CẤU TRÚC DÂN SỐ VÀ BÌNH ĐẲNG GIỚI

### 2.2. Mục tiêu 5: Phân tích tỷ lệ dân số nữ theo thời gian

**Tiêu chí SMART:**
- **S (Specific):** Phân tích sự thay đổi tỷ lệ dân số nữ tại 8 quốc gia.
- **M (Measurable):** Dựa trên biến `SP.POP.TOTL.FE.ZS` — tỷ lệ dân số nữ (% tổng dân số).
- **A (Achievable):** Data cực sạch, đầy đủ cho cả 8 quốc gia trong giai đoạn 2010–2023.
- **R (Relevant):** Phản ánh gender imbalance và social preference trong cấu trúc dân số.
- **T (Time-bound):** Từ năm 2010 đến 2023.

### Trường dữ liệu sử dụng

**`SP.POP.TOTL.FE.ZS`** — Population, female (% of total population)
- Tỷ lệ phần trăm dân số nữ so với tổng dân số.
- Mức **~50%** được coi là cân bằng tự nhiên.
- Sai lệch khỏi 50% có thể phản ánh: preference for sons, bất bình đẳng trong chăm sóc y tế, di cư, hoặc các yếu tố xã hội khác.

### Lý do lựa chọn metrics

Biến `SP.POP.TOTL.FE.ZS` phản ánh trực tiếp **cấu trúc giới** của dân số. Sự lệch khỏi mốc 50% (cân bằng sinh học tự nhiên) là tín hiệu của bất bình đẳng giới ở cấp độ **cấu trúc dân số học**, sâu sắc hơn nhiều so với các chỉ số bình đẳng giới khác.

In [ ]:
# ====== BIỂU ĐỒ 3: LINE CHART — Tỷ lệ dân số nữ theo thời gian ======

pop_df = scatter_df.copy()

plt.figure(figsize=(16, 8))

palette = sns.color_palette('tab10', n_colors=len(pop_df['Country Name'].unique()))
ax = sns.lineplot(
    data=pop_df,
    x='Year', y='female_pop',
    hue='Country Name',
    marker='o', markersize=6,
    palette=palette,
    linewidth=2.5
)

# Đường chuẩn 50%
ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Mốc cân bằng (50%)', zorder=0)

plt.title('Xu hướng tỷ lệ dân số nữ tại 8 quốc gia (2010–2023)',
          fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Năm', fontsize=13, fontweight='bold')
plt.ylabel('Tỷ lệ dân số nữ (%)', fontsize=13, fontweight='bold')
plt.xticks(range(2010, 2024))

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, title='Chú thích',
          bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

### Biểu đồ 4: Deviation Chart — Độ lệch tỷ lệ dân số nữ so với 50% (NÂNG CAO)

**Lý do chọn biểu đồ Deviation Chart:**
Thay vì nhìn giá trị tuyệt đối, Deviation Chart **highlight** mức độ bất bình đẳng cực rõ bằng cách tính:

`deviation = female_pop - 50`

- Giá trị **dương** → tỷ lệ nữ **cao hơn** bình thường
- Giá trị **âm** → tỷ lệ nữ **thấp hơn** bình thường (dấu hiệu gender imbalance)

Biểu đồ này giúp người xem **ngay lập tức nhận ra** quốc gia nào có vấn đề về bình đẳng giới trong cấu trúc dân số.

In [ ]:
# ====== BIỂU ĐỒ 4: DEVIATION CHART — Độ lệch so với 50% ======

# Tính deviation trung bình giai đoạn 2010-2023 cho mỗi quốc gia
dev_df = pop_df.groupby('Country Name')['female_pop'].mean().reset_index()
dev_df['deviation'] = dev_df['female_pop'] - 50
dev_df = dev_df.sort_values('deviation', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 7))

# Vẽ thanh deviation
colors = ['#e74c3c' if d < 0 else '#2ecc71' for d in dev_df['deviation']]
bars = ax.barh(dev_df['Country Name'], dev_df['deviation'], color=colors, height=0.6, edgecolor='white')

# Đường mốc 0
ax.axvline(x=0, color='black', linewidth=1.5, linestyle='-')

# Ghi chú giá trị
for i, (val, fem) in enumerate(zip(dev_df['deviation'], dev_df['female_pop'])):
    sign = '+' if val >= 0 else ''
    ax.text(val + (0.05 if val >= 0 else -0.05), i,
            f'{sign}{val:.2f}% (TB: {fem:.2f}%)',
            va='center', ha='left' if val >= 0 else 'right',
            fontsize=10, fontweight='bold',
            color='#27ae60' if val >= 0 else '#c0392b')

plt.title('Độ lệch tỷ lệ dân số nữ so với mốc cân bằng 50%\n(Trung bình giai đoạn 2010–2023)',
          fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Độ lệch so với 50% (percentage points)', fontsize=13, fontweight='bold')
plt.ylabel('')

# Thêm annotation
ax.text(0.98, 0.02, '← Thiếu nữ | Thừa nữ →',
        transform=ax.transAxes, fontsize=11, ha='right', va='bottom',
        style='italic', color='gray')

plt.tight_layout()
plt.show()

**Phân tích biểu đồ tỷ lệ dân số nữ:**

### 1. Nhóm quốc gia có tỷ lệ nữ cao hơn 50%:

- **Thailand (~50.6–51.3%)** và **Viet Nam (~51.0–51.4%):** Tỷ lệ nữ cao hơn nam, có xu hướng **ổn định hoặc tăng nhẹ ở Thái Lan** và **giảm nhẹ ở Việt Nam**.
  → Nguyên nhân có thể do tuổi thọ nữ cao hơn nam, tạo ra chênh lệch tích lũy trong dân số.

- **Japan (~51.1–51.2%)** và **Germany (~50.6–51.3%):** Tỷ lệ nữ cao, phản ánh tuổi thọ nữ cao và cấu trúc dân số già hóa. Đức có xu hướng **giảm dần** về phía 50%, cho thấy dân số đang cân bằng hơn.

### 2. Nhóm quốc gia có tỷ lệ nữ thấp hơn 50%:

- **India (~48.3–48.4%):** Tỷ lệ nữ **thấp nhất** trong nhóm, phản ánh hiện tượng **preference for sons** (ưu tiên con trai) lâu đời. Tuy nhiên, xu hướng **tăng nhẹ** cho thấy tình hình đang dần cải thiện.

- **Indonesia (~49.8%):** Gần mốc 50% nhất, cho thấy cấu trúc dân số **tương đối cân bằng**.

- **Korea (~49.2→50.1%):** Trường hợp **đặc biệt nhất** — từ tỷ lệ nữ thấp (49.2%) đã vượt qua mốc 50% vào khoảng 2019. Điều này phản ánh sự **cải thiện đáng kể** trong bình đẳng giới tại Hàn Quốc.

- **United States (~49.7–50.1%):** Gần mốc 50%, có xu hướng **giảm nhẹ** từ trên 50% xuống dưới 50%.

### 3. Insight tổng thể:

- **~50%** được coi là mức cân bằng sinh học tự nhiên. Sự lệch khỏi mốc này, đặc biệt ở **India** (thấp hơn 50%), là dấu hiệu rõ ràng của **bất bình đẳng giới** ở cấp độ cấu trúc dân số.

- Xu hướng chung cho thấy hầu hết các quốc gia đang **dần tiến về mốc cân bằng 50%**, cho thấy tình hình bình đẳng giới đang được cải thiện trên phạm vi toàn cầu.

### 4. Kết luận:

**Bất bình đẳng giới thể hiện ngay trong cấu trúc dân số.** Một số quốc gia (đặc biệt India) có imbalance rõ rệt, có thể do preference for sons và inequality trong chăm sóc y tế. Tuy nhiên, xu hướng cải thiện dần cho thấy các chính sách bình đẳng giới đang phát huy tác dụng.

# 3. Tổng kết phân tích sức khỏe phụ nữ và bình đẳng giới (2010–2023)

## 3.1. Hiệu quả chi tiêu y tế đối với sức khỏe phụ nữ (Mục tiêu 4)

- **Mối tương quan dương** giữa chi tiêu y tế và tuổi thọ nữ, nhưng **không tuyến tính hoàn hảo** và có hiện tượng **diminishing returns** rõ rệt.  
- **Mỹ là outlier lớn nhất:** chi tiêu cao nhất thế giới ($8,000–$13,000+/người) nhưng tuổi thọ nữ **thấp hơn** Nhật, Hàn, Đức → **inefficiency** trong hệ thống y tế.  
- **Nhật và Hàn** là mô hình hiệu quả nhất: chi tiêu vừa phải nhưng tuổi thọ nữ cao nhất → hệ thống y tế **toàn dân, chi phí hợp lý**.  
- Các nước đang phát triển (India, Vietnam, Indonesia) có **marginal returns cao**: chi tiêu nhỏ mang lại cải thiện tuổi thọ lớn.  

**💡 Kết luận:** Chi tiêu cao ≠ hiệu quả cao. Cần **phân bổ hiệu quả**, không chỉ tăng ngân sách.  

**👉 Policy:** Tập trung vào y tế dự phòng, chăm sóc sức khỏe ban đầu, và giảm bất bình đẳng trong tiếp cận dịch vụ y tế.

---

## 3.2. Cấu trúc dân số và bình đẳng giới (Mục tiêu 5)

- **India** có tỷ lệ nữ thấp nhất (~48.3%) → phản ánh **preference for sons** và bất bình đẳng giới sâu sắc trong cấu trúc dân số.  
- **Korea** có sự cải thiện ấn tượng nhất: từ 49.2% → 50.1% → **vượt qua mốc cân bằng 50%**.  
- Hầu hết quốc gia đang **dần tiến về mốc 50%**, cho thấy xu hướng cải thiện bình đẳng giới toàn cầu.  
- Các quốc gia có tỷ lệ nữ >50% (Nhật, Đức, Thái, Việt Nam) chủ yếu do **tuổi thọ nữ cao hơn nam**, không phải bất bình đẳng giới nghịch đảo.  

**💡 Kết luận:** Bất bình đẳng giới thể hiện ngay trong cấu trúc dân số. Xu hướng cải thiện, nhưng một số quốc gia vẫn còn **gender imbalance rõ rệt**.  

**👉 Gender:** Cần tiếp tục các chính sách bình đẳng giới, đặc biệt ở các quốc gia có tỷ lệ nữ thấp hơn 50%.

---

## 3.3. Kết luận tổng thể

1. **Chi tiêu y tế:** Có tương quan dương với tuổi thọ nữ nhưng hiệu suất giảm dần. Mỹ chi nhiều nhất nhưng hiệu quả kém nhất; Nhật, Hàn là mô hình tham khảo.  
2. **Cấu trúc dân số:** Đa số quốc gia hướng về cân bằng 50%, India có imbalance rõ nhất nhưng đang cải thiện.  
3. **Bài học chung:** Hiệu quả > Số lượng. Dù là chi tiêu y tế hay chính sách giới, **cách phân bổ và thực thi** mới là yếu tố quyết định kết quả.